In [2]:
import os
import json
import warnings

import pandas as pd
from datasets import load_dataset

warnings.filterwarnings("ignore")

PROCESSED_DIR = "processed"      
MERGED_DIR = "merged"
os.makedirs(MERGED_DIR, exist_ok=True)


In [3]:
REQUIRED_FIELDS = ["instruction", "input", "output", "metadata"]


def make_record(instruction, input_text, output_text, source, repo=None,
                 language=None, extra_metadata=None):
    metadata = {"source": source, "repo": repo, "language": language}
    if extra_metadata:
        metadata.update(extra_metadata)
    return {
        "instruction": instruction,
        "input": input_text,
        "output": output_text,
        "metadata": metadata,
    }


def validate_records(records, source_name):
    """Drop records missing required fields or with empty input/output."""
    valid = []
    dropped = 0
    for r in records:
        if not all(k in r for k in REQUIRED_FIELDS):
            dropped += 1
            continue
        if not r["input"] or not r["output"]:
            dropped += 1
            continue
        if not str(r["input"]).strip() or not str(r["output"]).strip():
            dropped += 1
            continue
        valid.append(r)
    print(f"[{source_name}] kept {len(valid)}, dropped {dropped} invalid/empty rows")
    return valid


In [4]:
def load_jsonl(path):
    records = []
    if not os.path.exists(path):
        print(f"Warning: {path} not found, skipping.")
        return records
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


github_records = []
for split in ["train", "val", "test"]:
    path = os.path.join(PROCESSED_DIR, f"{split}.jsonl")
    split_records = load_jsonl(path)
    for r in split_records:
        r["metadata"]["source"] = "github_pr"
        r["metadata"]["split_origin"] = split
    github_records.extend(split_records)

github_records = validate_records(github_records, "github_pr")
print(f"Total GitHub records: {len(github_records)}")


[github_pr] kept 329, dropped 0 invalid/empty rows
Total GitHub records: 329


In [8]:
NUTANIX_INSTRUCTION = (
    "You are an experienced software engineer performing a code review. "
    "Given the following code, write a concise, helpful review comment."
)

nutanix_records = []
try:
    ds = load_dataset("ronantakizawa/github-codereview", split="train")
    print("github-codereview columns:", ds.column_names)

    for row in ds:
        nutanix_records.append(make_record(
            instruction=NUTANIX_INSTRUCTION,
            input_text=row.get("before_code"),
            output_text=row.get("reviewer_comment"),
            source="github_codereview_diffs",
            language=row.get("language"),
            extra_metadata={
                "quality_score": row.get("quality_score"),
                "is_negative": row.get("is_negative"),
            },
        ))

except Exception as e:
    print(f"Could not load ronantakizawa/github-codereview: {e}")

nutanix_records = validate_records(nutanix_records, "github_codereview_diffs")
print(f"Total records: {len(nutanix_records)}")

README.md: 0.00B [00:00, ?B/s]

data/train/train-00000-of-00003.parquet:   0%|          | 0.00/92.9M [00:00<?, ?B/s]

data/train/train-00000-of-00004.parquet:   0%|          | 0.00/94.9M [00:00<?, ?B/s]

data/train/train-00001-of-00003.parquet:   0%|          | 0.00/92.7M [00:00<?, ?B/s]

data/train/train-00001-of-00004.parquet:   0%|          | 0.00/85.4M [00:00<?, ?B/s]

...

data/test/test-00000-of-00001.parquet:   0%|          | 0.00/19.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/334323 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10471 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11013 [00:00<?, ? examples/s]

github-codereview columns: ['before_code', 'reviewer_comment', 'after_code', 'diff_context', 'file_path', 'comment_line', 'language', 'quality_score', 'comment_type', 'comment_length', 'before_lines', 'after_lines', 'is_negative', 'pr_title', 'pr_number', 'repo_name', 'repo_stars', 'repo_language', 'reviewer_username', 'author_username']
[github_codereview_diffs] kept 334323, dropped 0 invalid/empty rows
Total records: 334323


In [9]:
dahoas_records = []
try:
    ds = load_dataset("Dahoas/code-review-instruct-critique-revision", split="train")
    print("Dahoas columns:", ds.column_names)

    instr_col = next((c for c in ["instruction", "prompt"] if c in ds.column_names), None)
    input_col = next((c for c in ["input", "code"] if c in ds.column_names), None)
    output_col = next((c for c in ["critique", "review", "output"] if c in ds.column_names), None)

    if output_col:
        for row in ds:
            instruction = (
                row.get(instr_col)
                if instr_col and row.get(instr_col)
                else "Review the following code and provide a critique with suggested improvements."
            )
            dahoas_records.append(make_record(
                instruction=instruction,
                input_text=row.get(input_col) if input_col else "",
                output_text=row.get(output_col),
                source="dahoas_critique_revision",
            ))
    else:
        print("Could not auto-detect output column -- "
              "inspect ds.column_names and update output_col manually.")

except Exception as e:
    print(f"Could not load Dahoas/code-review-instruct-critique-revision: {e}")

dahoas_records = validate_records(dahoas_records, "dahoas_critique_revision")
print(f"Total Dahoas records: {len(dahoas_records)}")


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-d3436bfb812be5(…):   0%|          | 0.00/184M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32800 [00:00<?, ? examples/s]

Could not load Dahoas/code-review-instruct-critique-revision: [{'expected': SplitInfo(name='train', num_bytes=322516541, num_examples=32800), 'recorded': SplitInfo(name='train', num_bytes=459184139, num_examples=45951, dataset_name='code-review-instruct-critique-revision')}]
[dahoas_critique_revision] kept 0, dropped 0 invalid/empty rows
Total Dahoas records: 0


In [12]:
CODEREVIEWQA_PATH = "google/code_x_glue_cc_code_refinement"

codereviewqa_records = []
try:
    ds = load_dataset(CODEREVIEWQA_PATH, "medium", split="train")
    print("Code refinement columns:", ds.column_names)

    for row in ds:
        codereviewqa_records.append(make_record(
            instruction=(
                "The following code contains a bug. Review it and provide "
                "the corrected version."
            ),
            input_text=row.get("buggy"),
            output_text=row.get("fixed"),
            source="codexglue_code_refinement",
        ))

except Exception as e:
    print(f"Could not load {CODEREVIEWQA_PATH}: {e}")

codereviewqa_records = validate_records(codereviewqa_records, "codexglue_code_refinement")
print(f"Total records: {len(codereviewqa_records)}")

README.md: 0.00B [00:00, ?B/s]

medium/train-00000-of-00001.parquet:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

medium/validation-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

medium/test-00000-of-00001.parquet:   0%|          | 0.00/1.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52364 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6545 [00:00<?, ? examples/s]

Code refinement columns: ['id', 'buggy', 'fixed']
[codexglue_code_refinement] kept 52364, dropped 0 invalid/empty rows
Total records: 52364


In [13]:
all_records = (
    github_records
    + nutanix_records
    + dahoas_records
    + codereviewqa_records
)

print(f"Total merged records (before dedup): {len(all_records)}")

df = pd.DataFrame(all_records)
df["source"] = df["metadata"].apply(lambda m: m.get("source"))

print("\nCounts by source:")
print(df["source"].value_counts())


Total merged records (before dedup): 387016

Counts by source:
source
github_codereview_diffs      334323
codexglue_code_refinement     52364
github_pr                       329
Name: count, dtype: int64


In [14]:
before = len(df)
df = df.drop_duplicates(subset=["input", "output"])
print(f"Rows before: {before}, after cross-source dedup: {len(df)} "
      f"(removed {before - len(df)})")


Rows before: 387016, after cross-source dedup: 274293 (removed 112723)


In [15]:
from sklearn.model_selection import train_test_split

def get_split_origin(metadata):
    return metadata.get("split_origin")

df["split_origin"] = df["metadata"].apply(get_split_origin)

has_origin = df[df["split_origin"].notna()]
no_origin = df[df["split_origin"].isna()]

train_no_origin, temp_no_origin = train_test_split(no_origin, test_size=0.2, random_state=42)
val_no_origin, test_no_origin = train_test_split(temp_no_origin, test_size=0.5, random_state=42)

train_df = pd.concat([has_origin[has_origin["split_origin"] == "train"], train_no_origin])
val_df = pd.concat([has_origin[has_origin["split_origin"] == "val"], val_no_origin])
test_df = pd.concat([has_origin[has_origin["split_origin"] == "test"], test_no_origin])

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")


Train: 219474
Val:   27402
Test:  27417


In [16]:
def save_jsonl(split_df, path):
    records = split_df[["instruction", "input", "output", "metadata"]].to_dict("records")
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} examples -> {path}")


save_jsonl(train_df, os.path.join(MERGED_DIR, "train.jsonl"))
save_jsonl(val_df, os.path.join(MERGED_DIR, "val.jsonl"))
save_jsonl(test_df, os.path.join(MERGED_DIR, "test.jsonl"))


Saved 219474 examples -> merged\train.jsonl
Saved 27402 examples -> merged\val.jsonl
Saved 27417 examples -> merged\test.jsonl


In [17]:
print("Final merged dataset:")
print(f"  Total examples: {len(df)}")
print(f"  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nBy source:")
print(df["source"].value_counts())

Final merged dataset:
  Total examples: 274293
  Train: 219474 | Val: 27402 | Test: 27417

By source:
source
github_codereview_diffs      221600
codexglue_code_refinement     52364
github_pr                       329
Name: count, dtype: int64
